# Experimento A — las TRES arquitecturas (EfficientNet-B0, ResNet-50, Xception)
### Antes de correr SE DEBE TENER EN UNA CARPETA ESTOS ARCHIVOS:
```
MyDrive/
    datos/config.py      <-- es el motor de entrenamiento y evaluación 
    datos/nucleo.py      <-- archivo de configuración centralizada
    StyleGAN2.zip         <-- El dataset de StyleGAN2
    Resultados/           <-- se crea sola
```

In [ ]:
import sys

print(sys.version)
print(sys.version_info)
print(sys.executable)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
sys.version_info(major=3, minor=13, micro=15, releaselevel='final', serial=0)
/usr/bin/python3


In [ ]:
!nvidia-smi

Sun Jul 19 02:02:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install timm --quiet
print("Listo")

Listo


In [ ]:

from google.colab import drive
import os, time, zipfile

drive.mount('/content/drive')

RUTA_ZIP_DRIVE = '/content/drive/MyDrive/Tesis/StyleGAN2_CelebA.zip'  # <-- AJUSTA
RUTA_DATOS_LOCAL = '/content/Particiones'  # disco local de la sesión (rápido)
GENERADOR_BASELINE = "StyleGAN2_CelebA"

if not os.path.exists(RUTA_ZIP_DRIVE):
    raise FileNotFoundError(
        f"No encuentro {RUTA_ZIP_DRIVE}\n"
        f"Revisa el nombre exacto del zip en Drive (mayúsculas incluidas)."
    )

destino = os.path.join(RUTA_DATOS_LOCAL, GENERADOR_BASELINE)

if not os.path.exists(destino):
    print("Descomprimiendo el dataset al disco local de Colab...")
    t0 = time.time()
    with zipfile.ZipFile(RUTA_ZIP_DRIVE, 'r') as z:
        z.extractall(RUTA_DATOS_LOCAL)
    print(f"Descomprimido en {time.time()-t0:.0f}s")
else:
    print("El dataset ya está descomprimido.")

if not os.path.isdir(os.path.join(destino, "train")):
    import shutil, glob
    candidatos = glob.glob(os.path.join(RUTA_DATOS_LOCAL, "**", GENERADOR_BASELINE, "train"),
                           recursive=True)
    if not candidatos:
        raise FileNotFoundError(
            f"Tras descomprimir no encuentro {GENERADOR_BASELINE}/train dentro de "
            f"{RUTA_DATOS_LOCAL}. Contenido: {os.listdir(RUTA_DATOS_LOCAL)}"
        )
    origen = os.path.dirname(candidatos[0])
    if os.path.abspath(origen) != os.path.abspath(destino):
        shutil.rmtree(destino, ignore_errors=True)
        shutil.move(origen, destino)

# Verificación: el Experimento A necesita los 3 splits con ambas clases
print("\nRuta de datos:", RUTA_DATOS_LOCAL)
total = 0
for split in ["train", "val", "test"]:
    for clase in ["fake", "real"]:
        carpeta = os.path.join(destino, split, clase)
        if not os.path.isdir(carpeta):
            raise FileNotFoundError(f"Falta la carpeta {carpeta}")
        n = len(os.listdir(carpeta))
        total += n
        print(f"  {GENERADOR_BASELINE}/{split}/{clase}: {n:,} imágenes")
print(f"  TOTAL: {total:,} imágenes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
El dataset ya está descomprimido.

Ruta de datos: /content/Particiones
  StyleGAN2_CelebA/train/fake: 4,900 imágenes
  StyleGAN2_CelebA/train/real: 4,900 imágenes
  StyleGAN2_CelebA/val/fake: 1,050 imágenes
  StyleGAN2_CelebA/val/real: 1,050 imágenes
  StyleGAN2_CelebA/test/fake: 1,050 imágenes
  StyleGAN2_CelebA/test/real: 1,050 imágenes
  TOTAL: 14,000 imágenes


In [ ]:
import os
print(os.listdir('/content/Particiones'))

['StyleGAN2_CelebA']


In [ ]:
import shutil, sys
from pathlib import Path

RUTA_CODIGO_DRIVE = Path('/content/drive/MyDrive/Tesis/codigo')  # <-- AJUSTA

for nombre in ['config.py', 'nucleo.py']:
    origen = RUTA_CODIGO_DRIVE / nombre
    if not origen.exists():
        raise FileNotFoundError(
            f"No encuentro {origen}\n"
            f"Sube tus {nombre} de la carpeta local del proyecto a {RUTA_CODIGO_DRIVE}/"
        )
    shutil.copy(origen, Path('/content') / nombre)

sys.path.insert(0, '/content')
import config
import nucleo

print("Motor importado desde Drive:")
print(f"  config.py y nucleo.py copiados de {RUTA_CODIGO_DRIVE}")

Motor importado desde Drive:
  config.py y nucleo.py copiados de /content/drive/MyDrive/Tesis/codigo


In [ ]:
from pathlib import Path

ARQUITECTURAS = ["efficientnet_b0", "resnet50", "legacy_xception"]

config.RUTA_PARTICIONES = Path(RUTA_DATOS_LOCAL)
config.RUTA_RESULTADOS  = Path('/content/drive/MyDrive/Tesis/Resultados2')
config.RUTA_MODELOS     = config.RUTA_RESULTADOS / "modelos"
config.RUTA_METRICAS    = config.RUTA_RESULTADOS / "metricas"
config.preparar_carpetas()

config.NUM_WORKERS = 4  

config.BATCH_SIZE = 16

print(f"Arquitecturas a entrenar: {ARQUITECTURAS}")
print(f"Dispositivo: {config.DISPOSITIVO} | Batch: {config.BATCH_SIZE}")
print("\nParámetros heredados de tu config.py (idénticos a local):")
print(f"  learning_rate ....... {config.LEARNING_RATE}")
print(f"  weight_decay ........ {config.WEIGHT_DECAY}")
print(f"  epocas_max .......... {config.EPOCAS}")
print(f"  paciencia ........... {config.PACIENCIA_EARLY_STOPPING}")
print(f"  preentrenado ........ {config.PREENTRENADO}")
print(f"  congelar_capas ...... {config.CONGELAR_CAPAS}")
print(f"  amp ................. {config.USAR_AMP}")
print(f"  aumento_robustez .... {config.AUMENTO_ROBUSTEZ}")
print(f"    blur .............. p={config.PROB_BLUR}, sigma={config.BLUR_SIGMA_RANGO}")
print(f"    jpeg .............. p={config.PROB_JPEG}, calidad={config.JPEG_CALIDAD_RANGO}")
print(f"  semillas ............ {config.SEMILLAS_ENTRENAMIENTO}")

if not config.AUMENTO_ROBUSTEZ:
    print("\n  [AVISO] AUMENTO_ROBUSTEZ = False en tu config.py: entrenarías SIN")
    print("  blur ni JPEG. Si quieres el pipeline anti-atajo, ponlo en True.")

Arquitecturas a entrenar: ['efficientnet_b0', 'resnet50', 'legacy_xception']
Dispositivo: cuda | Batch: 16

Parámetros heredados de tu config.py (idénticos a local):
  learning_rate ....... 0.0001
  weight_decay ........ 0.0001
  epocas_max .......... 20
  paciencia ........... 5
  preentrenado ........ True
  congelar_capas ...... False
  amp ................. True
  aumento_robustez .... True
    blur .............. p=0.5, sigma=(0.1, 2.5)
    jpeg .............. p=0.5, calidad=(30, 95)
  semillas ............ [42, 123, 2024]


In [ ]:
import json, statistics
from datetime import datetime
from pathlib import Path

GENERADOR_BASELINE = "StyleGAN2_CelebA"
RUTA_PARCIALES = config.RUTA_METRICAS / "parciales"
RUTA_PARCIALES.mkdir(parents=True, exist_ok=True)

resumen_final = {}   

for arquitectura in ARQUITECTURAS:
    config.MODELO = arquitectura

    print("\n" + "=" * 64)
    print(f"EXPERIMENTO A — {arquitectura} — {GENERADOR_BASELINE}")
    print("=" * 64)

    resultados = []
    for semilla in config.SEMILLAS_ENTRENAMIENTO:
        parcial = (RUTA_PARCIALES /
                   f"experimentoA_{arquitectura}_{GENERADOR_BASELINE}_semilla{semilla}.json")
        if parcial.exists():
            with open(parcial, encoding="utf-8") as f:
                resultados.append(json.load(f))
            print(f"  [semilla {semilla}] ya estaba hecha — la reutilizo")
            continue

        nombre = f"{arquitectura}_{GENERADOR_BASELINE}_semilla{semilla}.pth"
        ruta_pth = config.RUTA_MODELOS / nombre

        r = nucleo.entrenar_modelo(GENERADOR_BASELINE, semilla, ruta_pth)
        r["ruta_modelo"] = str(ruta_pth)
        resultados.append(r)

        with open(parcial, "w", encoding="utf-8") as f:
            json.dump(r, f, indent=2, ensure_ascii=False)
        print(f"    -> semilla {semilla} guardada en Drive")

    salida = {
        "experimento": "A_baseline",
        "modelo": arquitectura,
        "generador_entrenamiento": GENERADOR_BASELINE,
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "config": {
            "batch_size": config.BATCH_SIZE,
            "learning_rate": config.LEARNING_RATE,
            "weight_decay": config.WEIGHT_DECAY,
            "epocas_max": config.EPOCAS,
            "paciencia": config.PACIENCIA_EARLY_STOPPING,
            "preentrenado": config.PREENTRENADO,
            "amp": config.USAR_AMP,
        },
        "resultados_por_semilla": resultados,
    }
    ruta_json = config.RUTA_METRICAS / f"experimentoA_{arquitectura}_{GENERADOR_BASELINE}.json"
    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(salida, f, indent=2, ensure_ascii=False)

    accs = [r["metricas_test"]["accuracy"] for r in resultados]
    aucs = [r["metricas_test"]["auc"] for r in resultados]
    resumen_final[arquitectura] = (statistics.mean(accs), statistics.mean(aucs))

    print(f"  {'Semilla':>8} | {'Acc':>7} | {'AUC':>7} | {'F1':>7}")
    for r in resultados:
        m = r["metricas_test"]
        print(f"  {r['semilla']:>8} | {m['accuracy']:.4f} | {m['auc']:.4f} | "
              f"{m['f1_fake']:.4f}")
    print(f"  -> {ruta_json}")

print("\n" + "=" * 64)
print("TERMINADO — Experimento A para las tres arquitecturas")
print("=" * 64)
print(f"{'Arquitectura':<18} | {'Acc media':>10} | {'AUC media':>10}")
print("-" * 44)
for arq, (acc, auc) in resumen_final.items():
    print(f"{arq:<18} | {acc:>10.4f} | {auc:>10.4f}")


EXPERIMENTO A — efficientnet_b0 — StyleGAN2_CelebA


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            


  Entrenando en StyleGAN2_CelebA (semilla 42)
    Época  1 | loss train: 0.6442 | loss val: 0.0592 | acc train: 0.8085 | acc val: 0.9848
    Época  2 | loss train: 0.2862 | loss val: 0.0384 | acc train: 0.8909 | acc val: 0.9905
    Época  3 | loss train: 0.1882 | loss val: 0.0222 | acc train: 0.9246 | acc val: 0.9929
    Época  4 | loss train: 0.1587 | loss val: 0.0222 | acc train: 0.9413 | acc val: 0.9929
    Época  5 | loss train: 0.1193 | loss val: 0.0181 | acc train: 0.9530 | acc val: 0.9952
    Época  6 | loss train: 0.1057 | loss val: 0.0071 | acc train: 0.9597 | acc val: 0.9990
    Época  7 | loss train: 0.0864 | loss val: 0.0663 | acc train: 0.9686 | acc val: 0.9862
    Época  8 | loss train: 0.0834 | loss val: 0.0100 | acc train: 0.9701 | acc val: 0.9971
    Época  9 | loss train: 0.0797 | loss val: 0.0160 | acc train: 0.9734 | acc val: 0.9948
    Época 10 | loss train: 0.0633 | loss val: 0.0118 | acc train: 0.9761 | acc val: 0.9952
    Época 11 | loss train: 0.0523 | loss va

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            


  Entrenando en StyleGAN2_CelebA (semilla 42)
    Época  1 | loss train: 0.4229 | loss val: 0.0374 | acc train: 0.8040 | acc val: 0.9943
    Época  2 | loss train: 0.2494 | loss val: 0.0198 | acc train: 0.8933 | acc val: 0.9948
    Época  3 | loss train: 0.1959 | loss val: 0.0088 | acc train: 0.9182 | acc val: 0.9990
    Época  4 | loss train: 0.1673 | loss val: 0.0114 | acc train: 0.9327 | acc val: 0.9986
    Época  5 | loss train: 0.1367 | loss val: 0.0146 | acc train: 0.9456 | acc val: 0.9976
    Época  6 | loss train: 0.1160 | loss val: 0.0072 | acc train: 0.9541 | acc val: 0.9976
    Época  7 | loss train: 0.0953 | loss val: 0.0246 | acc train: 0.9637 | acc val: 0.9924
    Época  8 | loss train: 0.0817 | loss val: 0.0059 | acc train: 0.9695 | acc val: 0.9986
    Early stopping en época 8 (mejor acc val: 0.9990)
    Test -> acc: 0.7690 | auc: 0.9877 | f1(fake): 0.8121
    -> semilla 42 guardada en Drive

  Entrenando en StyleGAN2_CelebA (semilla 123)
    Época  1 | loss train: 0.4